# Tech Challenge Fase 2  
## Notebook 00 — Setup do Ambiente AWS S3 + Databricks + Unity Catalog

### Objetivo do notebook

Este notebook prepara o ambiente base do projeto no Databricks, utilizando o **bucket oficial AWS S3 `s3://s3tc2/`** por meio do **External Volume** já existente no Unity Catalog.

A partir desta versão, o projeto será isolado dentro de um prefixo corporativo e escalável:

```text
s3://s3tc2/projetos/fiap/tech_challenge_fase2/
```

No Databricks, esse caminho será acessado por:

```text
/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2
```

### Papel deste notebook na arquitetura

Este notebook não realiza ingestão nem transformação de dados. Ele é responsável por:

- validar o volume governado pelo Unity Catalog;
- criar a estrutura lógica do Data Lake;
- organizar arquivos enviados na raiz do volume para suas respectivas pastas `raw`;
- criar o arquivo central `config.json`;
- validar a existência dos arquivos esperados;
- registrar logs iniciais de validação.

### Arquitetura utilizada

```text
AWS S3
  ↓
Storage Credential
  ↓
External Location
  ↓
External Volume
  ↓
Raw
  ↓
Bronze
  ↓
Silver
  ↓
Gold
```

## 1. Validação do External Volume

Nesta etapa validamos o volume externo já criado no Unity Catalog.

Não criamos um novo volume, pois o bucket `s3://s3tc2/` já está coberto pelo volume:

```text
workspace.default.vol_trio_drive
```

Para evitar conflito de `LOCATION_OVERLAP`, utilizaremos uma **subpasta lógica** dentro desse volume, mantendo a nova versão do projeto isolada do histórico anterior.

In [0]:
SHOW VOLUMES IN workspace.default;

## 2. Conferência da localização do volume

Esta célula permite verificar se o volume `vol_trio_drive` está associado ao bucket oficial.

O resultado esperado é que o volume esteja relacionado ao caminho:

```text
s3://s3tc2/
```

Essa validação é importante para garantir que toda gravação feita no volume será refletida fisicamente na AWS S3.

In [0]:
%sql
DESCRIBE VOLUME workspace.default.vol_trio_drive;

## 3. Configuração central do projeto

Nesta etapa definimos os caminhos oficiais do projeto.

A decisão arquitetural adotada é utilizar o volume existente, porém gravar todos os artefatos dentro de um prefixo dedicado:

```text
projetos/fiap/tech_challenge_fase2
```

Essa organização permite que o bucket `s3://s3tc2/` cresça com outros projetos no futuro, sem misturar dados, logs ou versões de pipelines diferentes.

### Estrutura lógica definida

```text
/Volumes/workspace/default/vol_trio_drive/
└── projetos/
    └── fiap/
        └── tech_challenge_fase2/
            ├── raw/
            ├── bronze/
            ├── silver/
            ├── gold/
            ├── streaming/
            ├── logs/
            ├── docs/
            └── config/
```

In [0]:
%python
from datetime import datetime
import json

PROJECT_NAME = "fiap_alfabetizacao"
PROJECT_VERSION = "tech_challenge_fase2"

# Volume governado já existente no Unity Catalog.
VOLUME_ROOT = "/Volumes/workspace/default/vol_trio_drive"

# Prefixo lógico do projeto dentro do bucket oficial.
PROJECT_PREFIX = "projetos/fiap/tech_challenge_fase2"

# Caminho Databricks utilizado pelos notebooks.
BASE_PATH = f"{VOLUME_ROOT}/{PROJECT_PREFIX}"

# Caminho físico correspondente no Amazon S3.
BUCKET_S3 = "s3://s3tc2/projetos/fiap/tech_challenge_fase2/"

RAW_PATH = f"{BASE_PATH}/raw"
BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"
STREAMING_PATH = f"{BASE_PATH}/streaming"
LOG_PATH = f"{BASE_PATH}/logs"
DOCS_PATH = f"{BASE_PATH}/docs"
CONFIG_PATH = f"{BASE_PATH}/config"

EXECUTION_DATE = datetime.now().strftime("%Y-%m-%d")

print("Projeto:", PROJECT_NAME)
print("Versão:", PROJECT_VERSION)
print("Volume raiz:", VOLUME_ROOT)
print("Prefixo do projeto:", PROJECT_PREFIX)
print("BASE_PATH:", BASE_PATH)
print("BUCKET_S3:", BUCKET_S3)
print("Data de execução:", EXECUTION_DATE)

## 4. Criação da pasta principal do projeto

Nesta etapa criamos a subpasta isolada do projeto dentro do volume existente.

Na AWS S3, essa ação cria o prefixo lógico:

```text
s3://s3tc2/projetos/fiap/tech_challenge_fase2/
```

Caso a pasta já exista, o comando não sobrescreve dados existentes.

In [0]:
%python
dbutils.fs.mkdirs(BASE_PATH)

print("Pasta principal criada/validada com sucesso:")
print(BASE_PATH)

## 5. Criação da estrutura lógica do Data Lake

Nesta etapa criamos todas as pastas necessárias para o projeto.

A organização segue a Arquitetura Medalhão:

- `raw`: arquivos originais recebidos;
- `bronze`: dados ingeridos em formato analítico, preservando a origem;
- `silver`: dados padronizados, limpos e validados;
- `gold`: datasets analíticos prontos para Power BI e Machine Learning;
- `streaming`: dados e checkpoints da simulação de ingestão incremental;
- `logs`: registros de execução, qualidade e rejeições;
- `docs`: evidências e documentação técnica;
- `config`: arquivos de configuração e metadados do pipeline.

Também criamos a estrutura por domínio de dados:

```text
alunos
estados
municipios
metas_municipios
metas_ufs
```

In [0]:
%python
datasets = [
    "alunos",
    "estados",
    "municipios",
    "metas_municipios",
    "metas_ufs"
]

anos = [2023, 2024, 2025]

paths = [
    RAW_PATH,
    BRONZE_PATH,
    SILVER_PATH,
    GOLD_PATH,
    STREAMING_PATH,
    LOG_PATH,
    DOCS_PATH,
    CONFIG_PATH
]

# Estrutura RAW por domínio
for dataset in datasets:
    paths.append(f"{RAW_PATH}/{dataset}")

# Estrutura BRONZE por domínio e ano
for dataset in datasets:
    for ano in anos:
        paths.append(f"{BRONZE_PATH}/{dataset}/ano={ano}")

# Estrutura SILVER por domínio
for dataset in datasets:
    paths.append(f"{SILVER_PATH}/{dataset}")

# Estrutura GOLD analítica
gold_dirs = [
    "alfabetizacao_alunos",
    "alfabetizacao_estados",
    "alfabetizacao_municipios",
    "metas_municipios",
    "metas_ufs",
    "comparativo_meta_resultado",
    "ranking_uf",
    "ranking_municipio",
    "base_modelo_ia",
    "matriz_risco_educacional",
    "kpi_executivo",
    "feature_importance",
    "metricas_modelo_ia",
    "exports_powerbi"
]

for directory in gold_dirs:
    paths.append(f"{GOLD_PATH}/{directory}")

# Estrutura Streaming
streaming_dirs = [
    "input/indicadores_municipais",
    "checkpoint/bronze_indicadores_municipais",
    "checkpoint/silver_indicadores_municipais",
    "checkpoint/gold_indicadores_incrementais",
    "bronze_streaming/indicadores_municipais",
    "silver_streaming/indicadores_municipais",
    "gold_streaming/indicadores_incrementais"
]

for directory in streaming_dirs:
    paths.append(f"{STREAMING_PATH}/{directory}")

# Estrutura de logs
log_dirs = [
    "pipeline_execution/setup",
    "pipeline_execution/bronze",
    "pipeline_execution/silver",
    "pipeline_execution/gold",
    "data_quality/bronze",
    "data_quality/silver",
    "data_quality/gold",
    "streaming_metrics",
    "finops",
    "rejected",
    "rejected/alunos",
    "rejected/estados",
    "rejected/municipios",
    "rejected/metas_municipios",
    "rejected/metas_ufs"
]

for directory in log_dirs:
    paths.append(f"{LOG_PATH}/{directory}")

for path in paths:
    dbutils.fs.mkdirs(path)

print("Estrutura lógica criada/validada com sucesso.")

## 6. Definição dos arquivos esperados

Nesta etapa registramos todos os arquivos que devem estar disponíveis na camada `raw`.

Essa lista será usada para:

- orientar a organização automática dos arquivos;
- validar se todos os insumos foram enviados corretamente;
- gerar evidências de completude para a banca;
- alimentar os notebooks de ingestão Bronze.

Os arquivos devem ser organizados nos seguintes diretórios:

```text
raw/alunos/
raw/estados/
raw/municipios/
raw/metas_municipios/
raw/metas_ufs/
```

In [0]:
%python
expected_files = [
    {"dataset": "alunos", "ano": 2023, "file_name": "TS_ALUNO_2023.csv"},
    {"dataset": "alunos", "ano": 2024, "file_name": "TS_ALUNO_2024.csv"},
    {"dataset": "alunos", "ano": 2025, "file_name": "TS_ALUNO_2025.csv"},

    {"dataset": "estados", "ano": 2023, "file_name": "TS_ESTADO_2023.csv"},
    {"dataset": "estados", "ano": 2024, "file_name": "TS_ESTADO_2024.csv"},
    {"dataset": "estados", "ano": 2025, "file_name": "TS_ESTADO_2025.csv"},

    {"dataset": "municipios", "ano": 2023, "file_name": "TS_MUNICIPIO_2023.csv"},
    {"dataset": "municipios", "ano": 2024, "file_name": "TS_MUNICIPIO_2024.csv"},
    {"dataset": "municipios", "ano": 2025, "file_name": "TS_MUNICIPIO_2025.csv"},

    {"dataset": "metas_municipios", "ano": 2023, "file_name": "metas_municipios_2023.xlsx"},
    {"dataset": "metas_municipios", "ano": 2024, "file_name": "metas_municipios_2024.xlsx"},
    {"dataset": "metas_municipios", "ano": 2025, "file_name": "metas_municipios_2025.xlsx"},

    {"dataset": "metas_ufs", "ano": 2023, "file_name": "metas_ufs_2023.xlsx"},
    {"dataset": "metas_ufs", "ano": 2024, "file_name": "metas_ufs_2024.xlsx"},
    {"dataset": "metas_ufs", "ano": 2025, "file_name": "metas_ufs_2025.xlsx"}
]

for item in expected_files:
    item["expected_path"] = f"{RAW_PATH}/{item['dataset']}/{item['file_name']}"

display(spark.createDataFrame(expected_files))

## 7. Organização automática de arquivos enviados na raiz do projeto

Durante a etapa de upload no Databricks, é comum que os arquivos sejam enviados diretamente para a raiz do projeto.

Esta célula identifica arquivos na raiz e os move automaticamente para o subdiretório correto dentro de `raw`.

Exemplo:

```text
TS_ALUNO_2023.csv
↓
raw/alunos/TS_ALUNO_2023.csv
```

Essa etapa reduz erros manuais e facilita a preparação do pipeline.

In [0]:
%python
def identificar_dataset_por_nome(file_name: str):
    file_lower = file_name.lower()

    if file_lower.startswith("ts_aluno"):
        return "alunos"
    if file_lower.startswith("ts_estado"):
        return "estados"
    if file_lower.startswith("ts_municipio"):
        return "municipios"
    if file_lower.startswith("metas_municipios"):
        return "metas_municipios"
    if file_lower.startswith("metas_ufs"):
        return "metas_ufs"

    return None


arquivos_movidos = []

try:
    root_items = dbutils.fs.ls(BASE_PATH)

    for item in root_items:
        if item.isDir():
            continue

        dataset = identificar_dataset_por_nome(item.name)

        if dataset:
            origem = item.path
            destino = f"{RAW_PATH}/{dataset}/{item.name}"

            dbutils.fs.mv(origem, destino)

            arquivos_movidos.append({
                "file_name": item.name,
                "dataset": dataset,
                "origem": origem,
                "destino": destino,
                "status": "movido"
            })

except Exception as e:
    arquivos_movidos.append({
        "file_name": None,
        "dataset": None,
        "origem": BASE_PATH,
        "destino": None,
        "status": f"erro: {str(e)}"
    })

if arquivos_movidos:
    display(spark.createDataFrame(arquivos_movidos))
else:
    print("Nenhum arquivo encontrado na raiz do projeto para organização automática.")

## 8. Gravação do arquivo central de configuração

Nesta etapa gravamos o arquivo `config.json`.

Esse arquivo centraliza os caminhos e parâmetros do projeto, permitindo que os notebooks seguintes não tenham caminhos fixos no código.

Para compatibilidade com os notebooks já construídos e com a nova estrutura mais profissional, o arquivo contém:

- campos planos, como `raw_path`, `bronze_path`, `gold_path`;
- campos estruturados, como `project`, `environment`, `paths` e `datasets`.

Essa abordagem facilita a manutenção e permite evolução do projeto sem reescrever todos os notebooks.

In [0]:
%python
config = {
    # Campos planos para compatibilidade com notebooks existentes
    "project_name": PROJECT_NAME,
    "project_version": PROJECT_VERSION,
    "base_path": BASE_PATH,
    "raw_path": RAW_PATH,
    "bronze_path": BRONZE_PATH,
    "silver_path": SILVER_PATH,
    "gold_path": GOLD_PATH,
    "streaming_path": STREAMING_PATH,
    "log_path": LOG_PATH,
    "docs_path": DOCS_PATH,
    "config_path": CONFIG_PATH,
    "execution_date": EXECUTION_DATE,
    "bucket_s3": BUCKET_S3,
    "storage_credential": "trio-user",
    "external_volume": "workspace.default.vol_trio_drive",
    "volume_root": VOLUME_ROOT,
    "project_prefix": PROJECT_PREFIX,

    # Estrutura profissional do config
    "project": {
        "name": PROJECT_NAME,
        "version": PROJECT_VERSION,
        "execution_date": EXECUTION_DATE,
        "description": "FIAP Tech Challenge Fase 2 - Pipeline híbrido para análise da alfabetização no Brasil"
    },
    "environment": {
        "cloud": "AWS",
        "bucket_s3": BUCKET_S3,
        "volume_root": VOLUME_ROOT,
        "base_path": BASE_PATH,
        "external_volume": "workspace.default.vol_trio_drive",
        "unity_catalog": True
    },
    "paths": {
        "raw_path": RAW_PATH,
        "bronze_path": BRONZE_PATH,
        "silver_path": SILVER_PATH,
        "gold_path": GOLD_PATH,
        "streaming_path": STREAMING_PATH,
        "log_path": LOG_PATH,
        "docs_path": DOCS_PATH,
        "config_path": CONFIG_PATH
    },
    "datasets": {
        "names": datasets,
        "years": anos,
        "expected_files": expected_files
    }
}

config_file_path = f"{CONFIG_PATH}/config.json"

dbutils.fs.put(
    config_file_path,
    json.dumps(config, indent=4, ensure_ascii=False),
    overwrite=True
)

print("Configuração salva com sucesso em:")
print(config_file_path)

## 9. Validação da leitura do arquivo de configuração

Após salvar o `config.json`, realizamos uma leitura de validação.

Essa etapa confirma que o arquivo foi gravado corretamente e que os próximos notebooks poderão consumir as configurações de forma centralizada.

In [0]:
%python
config_loaded = json.loads(dbutils.fs.head(config_file_path))
config_loaded

## 10. Validação da estrutura criada

Esta etapa lista os principais diretórios criados no Data Lake.

O objetivo é permitir uma inspeção visual rápida e confirmar que:

- o prefixo do projeto foi criado;
- a camada Raw existe;
- a camada Bronze existe;
- a camada Silver existe;
- a camada Gold existe;
- os diretórios de logs e configuração estão disponíveis.

In [0]:
%python
principais_paths = {
    "BASE_PATH": BASE_PATH,
    "RAW_PATH": RAW_PATH,
    "BRONZE_PATH": BRONZE_PATH,
    "SILVER_PATH": SILVER_PATH,
    "GOLD_PATH": GOLD_PATH,
    "STREAMING_PATH": STREAMING_PATH,
    "LOG_PATH": LOG_PATH,
    "DOCS_PATH": DOCS_PATH,
    "CONFIG_PATH": CONFIG_PATH
}

for label, path in principais_paths.items():
    print("=" * 80)
    print(label, "->", path)

    try:
        display(dbutils.fs.ls(path))
    except Exception as e:
        print("Erro ao listar:", e)

## 11. Validação da presença dos arquivos esperados

Antes de iniciar a ingestão Bronze, validamos se todos os arquivos esperados estão presentes na camada `raw`.

Essa validação evita falhas posteriores nos notebooks de ingestão e gera um log rastreável da preparação do ambiente.

In [0]:
%python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

validacao_arquivos = []

for item in expected_files:
    expected_path = item["expected_path"]

    try:
        dbutils.fs.ls(expected_path)
        status = "OK"
        erro = ""
    except Exception as e:
        status = "PENDENTE"
        erro = str(e)

    validacao_arquivos.append({
        "dataset": str(item["dataset"]),
        "ano": int(item["ano"]),
        "file_name": str(item["file_name"]),
        "expected_path": str(expected_path),
        "status": str(status),
        "erro": str(erro)
    })

schema_validacao = StructType([
    StructField("dataset", StringType(), True),
    StructField("ano", IntegerType(), True),
    StructField("file_name", StringType(), True),
    StructField("expected_path", StringType(), True),
    StructField("status", StringType(), True),
    StructField("erro", StringType(), True)
])

df_validacao_arquivos = spark.createDataFrame(
    validacao_arquivos,
    schema=schema_validacao
)

display(df_validacao_arquivos.orderBy("dataset", "ano"))

## 12. Persistência do log de validação

O resultado da validação dos arquivos é salvo na camada de logs.

Essa prática permite auditoria da execução e evidencia para a banca que o projeto possui mecanismos básicos de observabilidade desde a etapa de setup.

In [0]:
%python
validation_path = f"{LOG_PATH}/pipeline_execution/setup/validacao_arquivos_execution_date={EXECUTION_DATE}"

dbutils.fs.mkdirs(f"{LOG_PATH}/pipeline_execution/setup")

(
    df_validacao_arquivos
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(validation_path)
)

print("Log de validação salvo em:")
print(validation_path)

## 13. Resumo da validação

Esta etapa apresenta um resumo consolidado por dataset e status.

O resultado esperado é que todos os arquivos apareçam como `OK`.

Caso algum arquivo esteja como `PENDENTE`, ele deverá ser enviado ou movido para o diretório correto antes da execução dos notebooks Bronze.

In [0]:
%python
df_resumo_validacao = (
    df_validacao_arquivos
    .groupBy("dataset", "status")
    .count()
    .orderBy("dataset", "status")
)

display(df_resumo_validacao)

## 14. Função opcional de pré-visualização de CSV

Esta função permite validar rapidamente o schema e uma pequena amostra de arquivos CSV.

Por padrão, ela **não executa contagem de linhas**, evitando custo desnecessário, especialmente para arquivos grandes como `TS_ALUNO`.

In [0]:
%python
def preview_csv(file_path, sep=";", encoding="ISO-8859-1", rows=5, infer_schema=True, do_count=False):
    reader = (
        spark.read
        .option("header", "true")
        .option("sep", sep)
        .option("encoding", encoding)
        .option("inferSchema", str(infer_schema).lower())
    )

    df = reader.csv(file_path)

    print("Arquivo:", file_path)
    print("Colunas:", len(df.columns))

    if do_count:
        print("Linhas:", df.count())

    df.printSchema()
    display(df.limit(rows))

    return df

## 15. Validação leve de arquivos CSV pequenos

Nesta etapa validamos uma pequena amostra dos arquivos de estados e municípios.

Essa validação é útil para confirmar separador, codificação e leitura do cabeçalho.

In [0]:
%python
csv_para_validar = [
    f"{RAW_PATH}/estados/TS_ESTADO_2023.csv",
    f"{RAW_PATH}/municipios/TS_MUNICIPIO_2023.csv"
]

for file_path in csv_para_validar:
    try:
        print("=" * 80)
        preview_csv(file_path=file_path, sep=";", encoding="ISO-8859-1", rows=3, do_count=False)
    except Exception as e:
        print("Não foi possível validar o arquivo:", file_path)
        print("Erro:", e)

## 16. Validação leve do arquivo de alunos

O arquivo de alunos costuma ser o maior conjunto da solução.

Por isso, validamos apenas o schema e uma pequena amostra, sem executar `count()`.

Essa prática reduz custo e tempo de execução em ambientes Serverless.

In [0]:
%python
try:
    preview_csv(
        file_path=f"{RAW_PATH}/alunos/TS_ALUNO_2023.csv",
        sep=";",
        encoding="ISO-8859-1",
        rows=5,
        infer_schema=True,
        do_count=False
    )
except Exception as e:
    print("Não foi possível validar TS_ALUNO_2023.csv.")
    print("Erro:", e)

## Resultado esperado

Ao final deste notebook, espera-se que:

- o External Volume esteja validado;
- a subpasta corporativa do projeto esteja criada;
- a estrutura `raw`, `bronze`, `silver`, `gold`, `streaming`, `logs`, `docs` e `config` exista;
- os arquivos enviados tenham sido organizados em `raw/<dataset>/`;
- o arquivo `config.json` tenha sido salvo;
- a validação dos arquivos esperados tenha sido registrada em logs.

### Próximo notebook

Após este setup, execute:

```text
01_bronze_orquestrador
```

Esse próximo notebook irá construir a tabela de metadados da ingestão Bronze e preparar os notebooks filhos responsáveis por cada dataset.